# 01b｜扩充英文通用与金融语料，复用原 8K 词表

130U-FinLLM｜130U 金融大语言模型

先保留已完成的 seed_v1 数据、词表和 300 步成果。本页创建独立的 expansion_v1：
固定 Wikipedia 快照默认最多 200MB，FOMC 纪要扩至 2011–2025；排除原 seed 文档与冻结任务文本，
按文档切分新增 train/dev，使用原 tokenizer 编码。实际独立篇数与 tokens 完成后报告。
不生成看多／看空标签，不重新训练词表，不启动模型训练。原文只保存在本运行时及自己的 Drive。

本页可用 CPU。每格按顺序执行；断开后先等重新连接，确认已停止再重新运行。
同版本重复运行先验证并复用完整产物；参数或输入变化时保留旧版本，另选新版本号。
下载和 Parquet 解码有额外缓存，不能把 200MB 输出上限当作全部内存／空间占用。
来源与权利说明见 docs/research/expanded_corpus_sources_20261011.md。

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} 与现有环境不兼容：{incompatible}")
    print("tokenizers 兼容性检查通过：", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"缺少 {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} 不满足 {requirement}")
    if problems:
        raise RuntimeError("项目依赖检查失败：" + "; ".join(problems))
    print("项目已安装依赖检查通过。")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"缺少项目 Python 源码：{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"当前 Notebook 导入了错误的项目位置：{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("当前 Notebook 的检查点恢复接口不可用。")
    print("当前 Notebook 内核导入检查通过：", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("新 Python 进程导入检查通过：", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
tokenizer_requirements = tokenizer_install_requirements()
print("本次 tokenizers 约束：", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".[data]",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("PyTorch 环境意外变化，请保留输出并停止训练。")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook 环境准备完成；PyTorch 保持：", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"/content 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
MOUNT_DRIVE = True
if not MOUNT_DRIVE:
    raise RuntimeError("本阶段需要已有种子输入及版本化备份，请完成 Drive 挂载。")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive 项目目录：", DRIVE_ROOT)

In [ ]:
EXPANSION_FETCH_SUCCEEDED = False
EXPANSION_CORPORA_BACKED_UP = False
EXPANSION_DATA_SUCCEEDED = False
EXPANSION_DATA_BACKED_UP = False
SEED_VERSION = "seed_v1"
EXPANSION_VERSION = "expansion_v1"
GENERAL_MAX_BYTES = 200_000_000
MAX_GENERAL_DOCUMENTS = 50_000
FINANCE_YEARS = ",".join(str(year) for year in range(2011, 2026))
FINANCE_MAX_BYTES = 12_000_000
MAX_FINANCE_DOCUMENTS = 128
from finllm.expansion_runtime import restore_expansion_materials, expansion_build_arguments, backup_expansion_data
from finllm.expansion import build_expansion_artifacts
expansion_paths = restore_expansion_materials(
    PROJECT, DRIVE_ROOT, EXPANSION_VERSION, SEED_VERSION)
print(json.dumps({key: str(value) for key, value in expansion_paths.items()}, indent=2, ensure_ascii=False))
print("本页只准备数据；当前 CUDA 可用：", torch.cuda.is_available())

获取限量候选。已完成的来源输出只读校验；下载中断时校验进度与页面缓存后继续。
不修改原 seed_v1。达到字节／篇数上限就停止，不承诺取得 100M 通用或 25M 独立金融 tokens。

In [ ]:
EXPANSION_FETCH_SUCCEEDED = False
EXPANSION_CORPORA_BACKED_UP = False
EXPANSION_DATA_SUCCEEDED = False
EXPANSION_DATA_BACKED_UP = False
command(sys.executable, "-u", "scripts/fetch_expansion_corpora.py",
        "--output", expansion_paths["corpora"],
        "--general-max-bytes", GENERAL_MAX_BYTES,
        "--max-general-documents", MAX_GENERAL_DOCUMENTS,
        "--finance-years", FINANCE_YEARS,
        "--finance-max-bytes", FINANCE_MAX_BYTES,
        "--max-finance-documents", MAX_FINANCE_DOCUMENTS, cwd=PROJECT)
print((expansion_paths["corpora"] / "manifest.json").read_text(encoding="utf-8"))
EXPANSION_FETCH_SUCCEEDED = True

In [ ]:
EXPANSION_CORPORA_BACKED_UP = False
EXPANSION_DATA_SUCCEEDED = False
EXPANSION_DATA_BACKED_UP = False
if not globals().get("EXPANSION_FETCH_SUCCEEDED", False):
    raise RuntimeError("来源获取尚未成功；先处理上方错误，保留已有输入。")
from finllm.runtime import sync_snapshot
source_backup = sync_snapshot(expansion_paths["corpora"], expansion_paths["drive"] / "corpora")
print(json.dumps(source_backup, indent=2, ensure_ascii=False))
EXPANSION_CORPORA_BACKED_UP = True
print("候选来源已校验并备份；随后开始清洗与编码。")

使用原 8K 词表；排除 seed 的训练与验证原文身份、文本重复／词法近重复，以及冻结任务池。
新增验证集仅用于当前语言模型诊断；原 seed 验证分片保留供 02c 比较训练前后损失。
这仍是无标签语言模型语料，不能据此报告资产立场准确率。

In [ ]:
EXPANSION_DATA_SUCCEEDED = False
EXPANSION_DATA_BACKED_UP = False
if not globals().get("EXPANSION_CORPORA_BACKED_UP", False):
    raise RuntimeError("请先成功备份候选来源，再构建扩充产物。")
expansion_data_result = build_expansion_artifacts(**expansion_build_arguments(expansion_paths))
print(json.dumps(expansion_data_result, indent=2, ensure_ascii=False))
print((expansion_paths["processed"] / "expansion_summary.json").read_text(encoding="utf-8"))
EXPANSION_DATA_SUCCEEDED = True

In [ ]:
EXPANSION_DATA_BACKED_UP = False
if not globals().get("EXPANSION_DATA_SUCCEEDED", False):
    raise RuntimeError("数据构建尚未成功；不得将半成品报告为完整备份。")
expansion_backup = backup_expansion_data(expansion_paths)
print(json.dumps(expansion_backup, indent=2, ensure_ascii=False))
shutil.copy2(PROJECT / "runtime_revision.txt", expansion_paths["drive"] / "runtime_revision.txt")
EXPANSION_DATA_BACKED_UP = True
print("扩充数据与校验备份完成：", expansion_paths["drive"])

完成标准：expansion_summary.json 记录真实新增文档／tokens；corpora、prepared、processed 三部分均 `verified: true`。
下一本 02c 改用 T4，显式承接已完成 seed 300 步的模型权重。不要重做 01a/02a/02b，也不要直接运行旧 03。
若本次候选不足以支持无回放的 610 步训练，02c 会停止并要求准备更多语料；不能用重复文本冒充新增数据。